In [1]:
import pandas as pd
from ete3 import Tree

In [13]:
three_data = "/home/ria/HSE/lab/new_three/data/old_data/knownpass/h3.l.r.newick"
aligned_data = "/home/ria/HSE/lab/new_three/data/old_data/knownpass/h3.all.fa"

In [14]:
# Чтение FASTA
def read_fasta(path):
    seqs = {}
    name = None
    buf = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if name is not None:
                    seqs[name] = "".join(buf)
                name = line[1:]
                buf = []
            else:
                buf.append(line)
        if name is not None:
            seqs[name] = "".join(buf)
    return seqs



In [15]:
seqs = read_fasta(aligned_data)
print(f"Прочитано последовательностей: {len(seqs)}")

Прочитано последовательностей: 5420


In [16]:
# Таблица трансляции кодонов
CODON2AA = {
    "TTT":"F","TTC":"F",
    "TTA":"L","TTG":"L","CTT":"L","CTC":"L","CTA":"L","CTG":"L",
    "ATT":"I","ATC":"I","ATA":"I",
    "ATG":"M",
    "GTT":"V","GTC":"V","GTA":"V","GTG":"V",
    "TCT":"S","TCC":"S","TCA":"S","TCG":"S",
    "CCT":"P","CCC":"P","CCA":"P","CCG":"P",
    "ACT":"T","ACC":"T","ACA":"T","ACG":"T",
    "GCT":"A","GCC":"A","GCA":"A","GCG":"A",
    "TAT":"Y","TAC":"Y",
    "CAT":"H","CAC":"H",
    "CAA":"Q","CAG":"Q",
    "AAT":"N","AAC":"N",
    "AAA":"K","AAG":"K",
    "GAT":"D","GAC":"D",
    "GAA":"E","GAG":"E",
    "TGT":"C","TGC":"C",
    "TGG":"W",
    "CGT":"R","CGC":"R","CGA":"R","CGG":"R","AGA":"R","AGG":"R",
    "AGT":"S","AGC":"S",
    "GGT":"G","GGC":"G","GGA":"G","GGG":"G",
    "TAA":"*","TAG":"*","TGA":"*",
}

In [17]:
# Трансляция всех последовательностей в аминокислотные

def translate_nuc_seq(nuc_seq):
    """Переводит нуклеотидную последовательность в аминокислотную."""
    aa = []
    for i in range(0, len(nuc_seq) - 2, 3):
        codon = nuc_seq[i:i+3].upper().replace("U", "T")
        if any(x not in "ATCG" for x in codon):
            aa.append("X")
        else:
            aa.append(CODON2AA.get(codon, "X"))
    return aa  # возвращаем список, чтобы обращаться по индексу

aa_seqs = {name: translate_nuc_seq(seq) for name, seq in seqs.items()}

# Проверка
any_aa = next(iter(aa_seqs.values()))
n_sites = len(any_aa)
print(f"Число аминокислотных сайтов: {n_sites}")

Число аминокислотных сайтов: 566


In [18]:
# Загрузка дерева и проверка

t=Tree(three_data, format=1)

missing = []
for n in t.traverse():
    if not n.name:
        missing.append("(unnamed node)")
    elif n.name not in seqs:
        missing.append(n.name)

print(f"Узлов без последовательности: {len(missing)}")
if missing:
    print("  ", missing[:10])

Узлов без последовательности: 1
   ['root']


In [19]:
#статистика по узлам
total_nodes = sum(1 for n in t.traverse())
leaves = sum(1 for n in t.iter_leaves())
internal = total_nodes - leaves

print(f"Всего узлов в дереве: {total_nodes}")
print(f"Листьев: {leaves}")
print(f"Внутренних: {internal}")
print(f"Последовательностей в fasta: {len(seqs)}")
print(f"Узлов без последовательности: {len(missing)}")

Всего узлов в дереве: 5421
Листьев: 2711
Внутренних: 2710
Последовательностей в fasta: 5420
Узлов без последовательности: 1


In [20]:
# форматируем названия, убераем суффикс после "/" в fasta

seqs = {name.split("/")[0]: seq for name, seq in seqs.items()}
aa_seqs = {name: translate_nuc_seq(seq) for name, seq in seqs.items()}

# Перепроверяем
missing = []
for n in t.traverse():
    if not n.name:
        missing.append("(unnamed)")
    elif n.name not in seqs:
        missing.append(n.name)
print(f"Узлов без последовательности после переименования: {len(missing)}")

Узлов без последовательности после переименования: 1


In [21]:
#основной

#  Поиск пар последовательных аминокислотных замен

def find_substitution_pairs(tree, aa_seqs, nuc_seqs, n_sites):
    """
    Обход дерева preorder с памятью по каждому сайту.
    
    Для каждого сайта запоминаем последнюю аминокислотную замену 
    на пути от корня. Когда встречаем новую замену в том же сайте —
    записываем пару (замена #1 → замена #2).
    
    Возвращает список словарей с колонками контрольной таблицы.
    """
    results = []
    
    def traverse(node, last_event):
        """
        last_event: dict {site_index: {
            'grandanc_aa': aa ДО предыдущей замены,
            'anc_aa':      aa ПОСЛЕ предыдущей замены,
            'grandanc_node': имя узла, где был grandanc,
            'anc_node':      имя узла, где появился anc
        }}
        """
        parent = node.up
        
        if parent is not None:
            parent_name = parent.name
            node_name = node.name
            
            parent_aa = aa_seqs.get(parent_name)
            node_aa = aa_seqs.get(node_name)
            
            if parent_aa is not None and node_aa is not None:
                for site_idx in range(n_sites):
                    aa_p = parent_aa[site_idx]
                    aa_n = node_aa[site_idx]
                    
                    # Пропускаем неинформативные или одинаковые
                    if aa_p in ("X", "*") or aa_n in ("X", "*"):
                        continue
                    if aa_p == aa_n:
                        continue
                    
                    # Есть аминокислотная замена на ребре parent→node в сайте site_idx
                    
                    # Если уже была замена раньше на этом пути — записываем пару
                    if site_idx in last_event:
                        prev = last_event[site_idx]
                        
                        # Подтягиваем кодоны из нуклеотидных последовательностей
                        nuc_start = site_idx * 3
                        
                        grandanc_nuc = nuc_seqs.get(prev['grandanc_node'])
                        grandanc_codon = grandanc_nuc[nuc_start:nuc_start+3] if grandanc_nuc else None
                        
                        anc_nuc = nuc_seqs.get(prev['anc_node'])
                        anc_codon = anc_nuc[nuc_start:nuc_start+3] if anc_nuc else None
                        
                        der_nuc = nuc_seqs.get(node_name)
                        der_codon = der_nuc[nuc_start:nuc_start+3] if der_nuc else None
                        
                        results.append({
                            'site':           site_idx + 1,       # 1-based
                            'ancestor_node':  prev["anc_node"],        # узел в котором появилась anc_aa
                            'grandanc_codon': grandanc_codon,
                            'grandanc_aa':    prev['grandanc_aa'],
                            'anc_codon':      anc_codon,
                            'anc_aa':         prev['anc_aa'],
                            'der_codon':      der_codon,
                            'der_aa':         aa_n,
                            'distance':       tree.get_distance(prev['anc_node'], node_name) - node.dist / 2,  # длина ребра от появления узла anc_aa до середины ребра der_aa
                            'distance_grand': tree.get_distance(prev['grandanc_node'], node_name) - node.dist / 2 # длина ребра от появления узла grandanc_aa до середины ребра der_aa
                        })
                    
                    # Обновляем last_event: текущая замена — теперь "последняя"
                    last_event[site_idx] = {
                        'grandanc_aa':   aa_p,          # состояние ДО текущей замены
                        'grandanc_node': parent_name,   # узел с этим состоянием
                        'anc_aa':        aa_n,          # состояние ПОСЛЕ текущей замены
                        'anc_node':      node_name      # узел где появился новый aa
                    }
        
        # Переходим к потомкам, у каждого своя копия памяти
        for child in node.children:
            traverse(child, last_event.copy())
    
    traverse(tree, {})
    return results

In [22]:
# создание DataFrame

events = find_substitution_pairs(t, aa_seqs, seqs, n_sites)
df = pd.DataFrame(events)

print(f"Найдено пар последовательных замен: {len(df)}")
print(f"Уникальных сайтов: {df['site'].nunique()}")
print()
print("Проверка: grandanc_aa != anc_aa во всех строках:", 
      (df['grandanc_aa'] != df['anc_aa']).all())
print("Проверка: anc_aa != der_aa во всех строках:", 
      (df['anc_aa'] != df['der_aa']).all())

print(df.shape)
df.head(10)

Найдено пар последовательных замен: 2492
Уникальных сайтов: 125

Проверка: grandanc_aa != anc_aa во всех строках: True
Проверка: anc_aa != der_aa во всех строках: True
(2492, 10)


,site,ancestor_node,grandanc_codon,grandanc_aa,anc_codon,anc_aa,der_codon,der_aa,distance,distance_grand
0,47,Node2626,AAT,N,GAT,D,AAT,N,0.003957,0.006127
1,138,Node2623,ACT,T,AAT,N,AGT,S,0.007799,0.024986
2,19,Node2622,CTT,L,TTT,F,CTT,L,0.009683,0.015533
3,209,Node2619,AGC,S,AAC,N,GAC,D,0.012955,0.017314
4,15,Node2618,CTC,L,TTC,F,CTC,L,0.018546,0.025356
5,233,Node2618,ATC,I,GTC,V,ATC,I,0.024762,0.031572
6,161,Node2617,AGC,S,AAC,N,AAA,K,0.028982,0.030946
7,99,Node2618,ACA,T,AAA,K,AGA,R,0.036041,0.042851
8,172,Node2615,AAA,K,GAA,E,AAA,K,0.014272,0.019360
9,14,Node2624,GCT,A,GTT,V,ATT,I,0.073282,0.076665


In [163]:

#  Сравнение с контрольной таблицей
df_old = pd.read_csv("/home/ria/HSE/lab/model/knownpass/h3_distances.csv")

print(f"Новая таблица: {len(df)} строк, {df['site'].nunique()} сайтов")
print(f"Контрольная:   {len(df_old)} строк, {df_old['site'].nunique()} сайтов")

# Общие сайты
old_sites = set(df_old['site'])
new_sites = set(df['site'])
print(f"\nСайты только в контрольной: {sorted(old_sites - new_sites)}")
print(f"Общих сайтов: {len(old_sites & new_sites)}")



Новая таблица: 2492 строк, 125 сайтов
Контрольная:   2492 строк, 125 сайтов

Сайты только в контрольной: []
Общих сайтов: 125


In [168]:
# сравнение distance
keys = ["site", "ancestor_node", "grandanc_codon", "grandanc_aa",
        "anc_codon", "anc_aa", "der_codon", "der_aa"]

# Нормализуем кодоны
for col in ["grandanc_codon", "anc_codon", "der_codon"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.upper()
    if col in df_old.columns:
        df_old[col] = df_old[col].astype(str).str.upper()

# Группируем по ключам, собираем отсортированные списки distance
old_sets = (df_old.groupby(keys, dropna=False)["distance"]
            .apply(lambda s: sorted(s.round(10).tolist()))
            .reset_index(name="distances_old"))

new_sets = (df.groupby(keys, dropna=False)["distance"]
            .apply(lambda s: sorted(s.round(10).tolist()))
            .reset_index(name="distances_new"))

cmp = pd.merge(new_sets, old_sets, on=keys, how="outer", indicator=True)

print("Только в new:", (cmp["_merge"] == "left_only").sum())
print("Только в old:", (cmp["_merge"] == "right_only").sum())
print("В обоих:", (cmp["_merge"] == "both").sum())

# Сравниваем списки distance с допуском
both = cmp[cmp["_merge"] == "both"].copy()

if len(both) > 0:
    def lists_close(a, b, tol=1e-6):
        if len(a) != len(b):
            return False
        return all(abs(x - y) < tol for x, y in zip(a, b))

    both["match"] = [lists_close(a, b) for a, b in zip(both["distances_new"], both["distances_old"])]

    print(f"\nСовпавших ключей: {len(both)}")
    print(f"Distances совпадают: {sum(both['match'])}")
    print(f"Distances НЕ совпадают: {sum(not m for m in both['match'])}")

    bad = both[~pd.Series(both["match"].tolist(), index=both.index)]
    if len(bad) > 0:
        print("\nПримеры несовпадений:")
        print(bad.head(10).to_string(index=False))
else:
    print("\nНет совпавших ключей — проверьте типы данных:")
    print("df types:", df[keys].dtypes.to_dict())
    print("df_old types:", df_old[keys].dtypes.to_dict())
    print("\ndf пример:", df[keys].head(2).to_string(index=False))
    print("df_old пример:", df_old[keys].head(2).to_string(index=False))

Только в new: 0
Только в old: 0
В обоих: 620

Совпавших ключей: 620
Distances совпадают: 620
Distances НЕ совпадают: 0


In [23]:
df.to_csv("/home/ria/HSE/lab/new_three/data/old_data/old_data_granddistance/old_data_h3_granddistances.csv", sep=",", index=False)